# Day 5-2: MobileNetV2 Transfer Learning

**강의 시간**: 2시간  
**학습 목표**:
- MobileNetV2 아키텍처 이해 (Depthwise Separable Conv)
- Transfer Learning 2-Phase 전략
- Data Augmentation 적용
- 경량 모델로 97%+ 정확도 달성

**사전 요구사항**: Day 5-1 완료  
**목표 성능**: Accuracy 97%+, Latency < 30ms

## 🔧 0. 환경 설정

In [ ]:
# 라이브러리 설치
%pip install -q 'mlflow>=2,<3' dagshub tensorflow opencv-python scikit-learn

print("✅ 라이브러리 설치 완료!")

In [ ]:
# 라이브러리 임포트
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import os
import time
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

import mlflow
import dagshub

np.random.seed(42)
tf.random.set_seed(42)

print(f"✅ TensorFlow {tf.__version__}")
print(f"✅ GPU: {len(tf.config.list_physical_devices('GPU'))} devices")

In [ ]:
# 시각화 설정
sns.set_style('whitegrid')

!wget -q -O NanumGothic.ttf -L "https://fonts.gstatic.com/ea/nanumgothic/v5/NanumGothic-Regular.ttf"

import matplotlib.font_manager as fm
font_path = "NanumGothic.ttf"
fm.fontManager.addfont(font_path)
plt.rcParams['figure.dpi'] = 300
plt.rcParams['savefig.dpi'] = 300
font_prop = fm.FontProperties(fname=font_path)
plt.rcParams["font.family"] = font_prop.get_name()
plt.rcParams["axes.unicode_minus"] = False

In [ ]:
# MLflow 설정
import mlflow
import dagshub

# 🔥 본인의 정보로 수정!
repo_owner = # 🔥 직접 작성이 필요합니다.
repo_name  = # 🔥 직접 작성이 필요합니다.

dagshub.init(repo_owner=repo_owner, repo_name=repo_name, mlflow=True)
mlflow.set_experiment('day5-gesture-recognition')

print("✅ MLflow 설정 완료!")

## 📂 1. 데이터 로딩 (Day 5-1과 동일)

### HaGRID 데이터 다운로드

**Day 5-1에서 이미 다운로드했다면** 아래 셀을 실행하지 않아도 됩니다.
데이터가 없다면 Day 5-1 노트북을 먼저 실행하세요.

In [ ]:
import os
from google.colab import userdata

os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_API_TOKEN')
os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')

try:
    from kaggle.api.kaggle_api_extended import KaggleApi

    api = KaggleApi()
    api.authenticate()

    dataset_id = 'innominate817/hagrid-classification-512p-no-gesture-150k'
    print(f"📥 {dataset_id} 다운로드 시작...")

    api.dataset_download_files(
        dataset_id,
        path='./data',
        unzip=True,
        quiet=False
    )

    print("\n✅ 다운로드 및 압축 해제 완료!")

except Exception as e:
    print(f"\n❌ 오류 발생: {e}")

In [ ]:
# 데이터 경로 확인 & 분할
data_dir = Path('./data/hagrid-classification-512p-no-gesture-150k')

if not data_dir.exists():
    print("❌ 데이터 폴더가 없습니다! Day 5-1을 먼저 실행하세요.")
else:
    gesture_folders = sorted([d for d in data_dir.iterdir() if d.is_dir()])
    gesture_names = [f.name for f in gesture_folders]
    print(f"✅ {len(gesture_names)}개 클래스 확인")

all_paths = []
all_labels = []
gesture_to_idx = {name: idx for idx, name in enumerate(gesture_names)}

for gesture_name in gesture_names:
    gesture_folder = data_dir / gesture_name
    image_paths = list(gesture_folder.glob('*.jpeg'))
    for img_path in image_paths:
        all_paths.append(str(img_path))
        all_labels.append(gesture_to_idx[gesture_name])

print(f"✅ 총 {len(all_paths):,}개 이미지")

# Train/Val/Test Split (70/15/15)
train_paths, temp_paths, train_labels, temp_labels = train_test_split(
    all_paths, all_labels, test_size=0.3, stratify=all_labels, random_state=42
)
val_paths, test_paths, val_labels, test_labels = train_test_split(
    temp_paths, temp_labels, test_size=0.5, stratify=temp_labels, random_state=42
)

print(f"Train: {len(train_paths):,}개 (70%)")
print(f"Val  : {len(val_paths):,}개 (15%)")
print(f"Test : {len(test_paths):,}개 (15%)")

## 🎨 2. Data Augmentation

In [ ]:
# Data Augmentation 레이어
# 🔥 4가지 augmentation을 직접 채워보세요:
#   RandomFlip('horizontal'), RandomRotation(0.1), RandomZoom(0.1), RandomTranslation(0.1, 0.1)
data_augmentation = keras.Sequential([
    # 🔥 직접 작성이 필요합니다. (layers.RandomFlip('horizontal'))
    # 🔥 직접 작성이 필요합니다. (layers.RandomRotation(0.1))
    # 🔥 직접 작성이 필요합니다. (layers.RandomZoom(0.1))
    # 🔥 직접 작성이 필요합니다. (layers.RandomTranslation(0.1, 0.1))
], name='data_augmentation')

print("✅ Data Augmentation 레이어 생성!")

In [ ]:
# Augmentation 시각화
sample_img_path = all_paths[0]
sample_img = cv2.imread(sample_img_path)
sample_img = cv2.cvtColor(sample_img, cv2.COLOR_BGR2RGB)
sample_img = cv2.resize(sample_img, (224, 224))
sample_img = sample_img / 255.0

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

axes[0].imshow(sample_img)
axes[0].set_title('Original', fontweight='bold')
axes[0].axis('off')

for i in range(1, 8):
    aug_img = data_augmentation(sample_img[np.newaxis, ...], training=True)[0]
    axes[i].imshow(aug_img)
    axes[i].set_title(f'Augmented {i}', fontweight='bold')
    axes[i].axis('off')

plt.suptitle('Data Augmentation 예시', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 📊 3. tf.data.Dataset 생성 (MobileNetV2용)

In [ ]:
# MobileNetV2용 전처리 함수
# 핵심: Baseline CNN의 img/255.0 대신 preprocess_input 사용!
def load_and_preprocess_mobilenet(path, label):
    """MobileNetV2용 전처리 — ImageNet 기준 [-1, 1] 범위로 변환"""
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, (224, 224))
    img = # 🔥 직접 작성이 필요합니다. (preprocess_input(img))  ← [0,255] → [-1, 1]
    return img, label

print("✅ MobileNetV2 Preprocessing 함수 정의!")
print("   - 512×512 → 224×224 리사이즈")
print("   - preprocess_input: [0,255] → [-1, 1]")
print("   - Baseline CNN(0~1)과 달리 반드시 preprocess_input 사용!")

In [ ]:
# tf.data.Dataset 생성
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

# Train Dataset (with augmentation)
train_dataset = tf.data.Dataset.from_tensor_slices((train_paths, train_labels))
train_dataset = train_dataset.map(load_and_preprocess_mobilenet, num_parallel_calls=AUTOTUNE)
train_dataset = train_dataset.shuffle(1000).batch(BATCH_SIZE)

def augment_batch(images, labels):
    """배치 단위로 augmentation 적용"""
    images = data_augmentation(images, training=True)
    return images, labels

train_dataset = train_dataset.map(augment_batch, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)

# Val/Test Dataset (no augmentation)
val_dataset = tf.data.Dataset.from_tensor_slices((val_paths, val_labels))
val_dataset = val_dataset.map(load_and_preprocess_mobilenet, num_parallel_calls=AUTOTUNE)
val_dataset = val_dataset.batch(BATCH_SIZE).prefetch(AUTOTUNE)

test_dataset = tf.data.Dataset.from_tensor_slices((test_paths, test_labels))
test_dataset = test_dataset.map(load_and_preprocess_mobilenet, num_parallel_calls=AUTOTUNE)
test_dataset = test_dataset.batch(BATCH_SIZE).prefetch(AUTOTUNE)

print("✅ Dataset 생성 완료!")
print(f"   Train batches: {len(train_dataset)} (with augmentation)")
print(f"   Val batches: {len(val_dataset)}")
print("\n💡 shuffle → batch → augment 순서 중요!")

## 🏗️ 4. MobileNetV2 모델 구축

In [ ]:
# MobileNetV2 Base Model (Phase 1: Frozen)
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)
base_model.trainable = False   # Phase 1: 완전 동결

print("✅ MobileNetV2 Base Model 로드 완료!")
print(f"   Pretrained: ImageNet")
print(f"   Trainable: {base_model.trainable} (Phase 1 — Frozen ❄️)")
print(f"   Parameters: {base_model.count_params():,}")

In [ ]:
# Custom Classifier 추가
inputs = keras.Input(shape=(224, 224, 3))

# Base model (training=False → BN이 ImageNet 통계 사용)
x = base_model(inputs, training=False)

# Classifier head
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(len(gesture_names), activation='softmax')(x)   # 19 classes

model = Model(inputs, outputs, name='MobileNetV2_Gesture')

print("\n✅ 전체 모델 구축 완료!")
print(f"   Total Parameters: {model.count_params():,}")
print(f"   Trainable Parameters (Phase 1): {sum([tf.size(w).numpy() for w in model.trainable_weights]):,}")

## 🏃 5. Phase 1: Feature Extraction (Base Frozen)

In [ ]:
# Phase 1 컴파일
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("✅ Phase 1 컴파일 완료!")
print("   Base Model: Frozen ❄️")
print("   Learning Rate: 1e-3")
print("   Epochs: 5")

In [ ]:
# Callbacks
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=3,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=2,
    min_lr=1e-7
)

print("✅ Callbacks 설정 완료!")

In [ ]:
# Phase 1 학습
try:
    mlflow.end_run()
except:
    pass

with mlflow.start_run(run_name=''):  # 🔥 직접 작성이 필요합니다. (예: 'MobileNetV2_Phase1_FeatureExtraction')
    mlflow.log_params({
        'model': 'MobileNetV2',
        'phase': 'feature_extraction',
        'pretrained': 'ImageNet',
        'base_trainable': False,
        'input_size': 224,
        'batch_size': BATCH_SIZE,
        'learning_rate': 1e-3,
        'epochs': 5,
        'optimizer': 'Adam',
        'augmentation': True
    })

    print("🏃 Phase 1 학습 시작...\n")
    # ⏱️ GPU에 따라 5~10분 소요

    history_phase1 = model.fit(
        train_dataset,
        epochs=5,
        validation_data=val_dataset,
        callbacks=[early_stop, reduce_lr],
        verbose=1
    )

    final_loss, final_acc = model.evaluate(val_dataset, verbose=0)

    mlflow.log_metrics({
        'phase1_final_val_loss': final_loss,
        'phase1_final_val_accuracy': final_acc
    })

    print(f"\n{'='*60}")
    print("  Phase 1 학습 완료")
    print('='*60)
    print(f"  Val Accuracy: {final_acc:.4f} ({final_acc*100:.2f}%)")
    print('='*60)

## 🔥 6. Phase 2: Fine-tuning (Base Unfrozen)

In [ ]:
# 🔥 Base model unfreeze (Fine-tuning 시작!)
# Pretrained 가중치를 조금씩 업데이트합니다
base_model.trainable = # 🔥 직접 작성이 필요합니다. (True)

print(f"🔓 Base Model Unfrozen!")
print(f"   Total Parameters: {model.count_params():,}")
print(f"   Trainable Parameters: {sum([tf.size(w).numpy() for w in model.trainable_weights]):,}")

In [ ]:
# Phase 2 컴파일
# 🔥 LR을 1e-5로 설정하세요 (Phase 1의 1/100 — Pretrained 가중치 보호)
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=# 🔥 직접 작성이 필요합니다.),  # 100배 낮은 LR!
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("✅ Phase 2 컴파일 완료!")
print("   Base Model: Unfrozen 🔥")
print("   Learning Rate: 1e-5 (Phase 1 대비 100배 감소)")
print("   Epochs: 10")

In [ ]:
# Phase 2 학습
try:
    mlflow.end_run()
except:
    pass

with mlflow.start_run(run_name=''):  # 🔥 직접 작성이 필요합니다. (예: 'MobileNetV2_Phase2_FineTuning')
    mlflow.log_params({
        'model': 'MobileNetV2',
        'phase': 'fine_tuning',
        'pretrained': 'ImageNet',
        'base_trainable': True,
        'input_size': 224,
        'batch_size': BATCH_SIZE,
        'learning_rate': 1e-5,
        'epochs': 10,
        'optimizer': 'Adam',
        'augmentation': True
    })

    print("🏃 Phase 2 학습 시작...\n")
    # ⏱️ GPU에 따라 20~30분 소요

    history_phase2 = model.fit(
        train_dataset,
        epochs=10,
        validation_data=val_dataset,
        callbacks=[early_stop, reduce_lr],
        verbose=1
    )

    final_loss, final_acc = model.evaluate(val_dataset, verbose=0)

    mlflow.log_metrics({
        'phase2_final_val_loss': final_loss,
        'phase2_final_val_accuracy': final_acc
    })

    model.save('mobilenetv2_gesture_final.keras')
    mlflow.keras.log_model(model, 'model')

    model_size_mb = os.path.getsize('mobilenetv2_gesture_final.keras') / (1024**2)
    mlflow.log_metric('model_size_mb', model_size_mb)

    print(f"\n{'='*60}")
    print("  Phase 2 학습 완료")
    print('='*60)
    print(f"  Val Accuracy: {final_acc:.4f} ({final_acc*100:.2f}%)")
    print(f"  Model Size: {model_size_mb:.2f} MB")
    print('='*60)

## 📈 7. 학습 곡선 분석

In [ ]:
# Phase 1 + Phase 2 결합 학습 곡선
phase1_epochs = len(history_phase1.history['loss'])
phase2_epochs = len(history_phase2.history['loss'])

train_loss = history_phase1.history['loss'] + history_phase2.history['loss']
val_loss = history_phase1.history['val_loss'] + history_phase2.history['val_loss']
train_acc = history_phase1.history['accuracy'] + history_phase2.history['accuracy']
val_acc = history_phase1.history['val_accuracy'] + history_phase2.history['val_accuracy']

epochs_range = range(1, phase1_epochs + phase2_epochs + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs_range, train_loss, 'b-', label='Train Loss', linewidth=2)
axes[0].plot(epochs_range, val_loss, 'r-', label='Val Loss', linewidth=2)
axes[0].axvline(phase1_epochs, color='gray', linestyle='--', linewidth=2, label='Phase 1→2')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('학습 곡선 - Loss', fontweight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(epochs_range, train_acc, 'b-', label='Train Accuracy', linewidth=2)
axes[1].plot(epochs_range, val_acc, 'r-', label='Val Accuracy', linewidth=2)
axes[1].axvline(phase1_epochs, color='gray', linestyle='--', linewidth=2, label='Phase 1→2')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('학습 곡선 - Accuracy', fontweight='bold')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.suptitle('MobileNetV2 2-Phase Training', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Phase 1 (Frozen): {phase1_epochs} epochs")
print(f"Phase 2 (Unfrozen): {phase2_epochs} epochs")
print(f"Final Val Accuracy: {val_acc[-1]:.4f} ({val_acc[-1]*100:.2f}%)")

## 📊 8. 최종 모델 평가

In [ ]:
# Test set 예측
print("🔮 Test set 예측 중...")

y_true = []
y_pred = []

for images, labels in test_dataset:
    preds = model.predict(images, verbose=0)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)

test_acc = np.mean(y_true == y_pred)
print(f"\n✅ Test Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")

In [ ]:
# Confusion Matrix (전체 19 클래스)
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(14, 12))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=gesture_names, yticklabels=gesture_names)
plt.xlabel('예측', fontweight='bold', fontsize=12)
plt.ylabel('실제', fontweight='bold', fontsize=12)
plt.title('Confusion Matrix — MobileNetV2', fontweight='bold', fontsize=14)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Classification Report
report = classification_report(y_true, y_pred, target_names=gesture_names, digits=4)
print("="*80)
print("  Classification Report — MobileNetV2")
print("="*80)
print(report)
print("="*80)

## ⚡ 9. 추론 속도 측정

In [ ]:
# Latency 측정
print("⏱️ Latency 측정 중...")

# Warmup
for images, _ in test_dataset.take(3):
    _ = model.predict(images, verbose=0)

latencies = []
for images, _ in test_dataset.take(20):
    start = time.time()
    _ = model.predict(images, verbose=0)
    latency = (time.time() - start) * 1000 / BATCH_SIZE
    latencies.append(latency)

avg_latency = np.mean(latencies)
fps = 1000 / avg_latency

print("\n" + "="*60)
print("  추론 속도")
print("="*60)
print(f"Avg Latency: {avg_latency:.2f} ms/image")
print(f"FPS: {fps:.1f}")
print(f"Model Size: {model_size_mb:.2f} MB")
print("="*60)

print(f"\n💡 목표 달성 여부:")
print(f"   Latency < 30ms: {'✅' if avg_latency < 30 else '❌'} ({avg_latency:.2f}ms)")
print(f"   Accuracy > 90%: {'✅' if final_acc > 0.90 else '❌'} ({final_acc*100:.2f}%)")

## 📊 10. Baseline vs MobileNetV2 비교

In [ ]:
# 성능 비교표
comparison = pd.DataFrame({
    'Model': ['Baseline CNN', 'MobileNetV2'],
    'Parameters': ['~133K', '~2.4M'],
    'Size (MB)': [1.6, model_size_mb],
    'Val Accuracy (%)': [73.52, final_acc * 100],
    'Test Accuracy (%)': ['-', f'{test_acc * 100:.2f}'],
    'Latency (ms)': [2.0, avg_latency],
    'FPS': [497, fps]
})

print("="*80)
print("  Baseline CNN vs MobileNetV2")
print("="*80)
print(comparison.to_string(index=False))
print("="*80)

print(f"\n개선:")
print(f"  Accuracy: +{(final_acc - 0.7352) * 100:.2f}%p")
print(f"  Latency: 유사 (MobileNetV2가 더 빠름)")

## ✅ Day 5-2 완료 체크리스트

- [ ] Depthwise Separable Conv 원리 이해 (~1/7 연산량)
- [ ] Inverted Residual Block 개념 이해 (Narrow→Wide→Narrow)
- [ ] Data Augmentation 4가지 직접 구현
- [ ] MobileNetV2용 `preprocess_input` 적용 ([-1, 1] 범위)
- [ ] Phase 1: Feature Extraction (lr=1e-3, 5 epochs, base frozen)
- [ ] Phase 2: `base_model.trainable = True` 설정
- [ ] Phase 2: lr=1e-5로 컴파일 (100배 감소 이유 이해)
- [ ] Phase 2: Fine-tuning (10 epochs)
- [ ] Phase 1+2 결합 학습 곡선 시각화
- [ ] Val Accuracy 97%+ 달성
- [ ] Latency ~1.8ms, FPS ~556 확인
- [ ] MLflow에 Phase 1, Phase 2 각각 기록

## 🎯 다음 단계 (Day 5-3)

**모델 최적화 & 배포**
- TFLite 변환
- INT8 Quantization (~1/4 크기 감소)
- 실시간 제스처 데모